In [1]:
!git clone https://github.com/pkuliyi2015/GeoBloom.git
%cd GeoBloom
!conda install -y -c conda-forge cuda-cudart
!pip install jieba_fast xxhash==3.4.1 scikit-learn==1.4.2
!g++ nnue/v19/nnue.cpp -o nnue/v19/nnue -pthread -mavx2 -O3 -fno-tree-vectorize

Cloning into 'GeoBloom'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 313 (delta 15), reused 20 (delta 6), pack-reused 276 (from 1)
Receiving objects: 100% (313/313), 196.66 MiB | 38.51 MiB/s, done.
Resolving deltas: 100% (145/145), done.
/kaggle/working/GeoBloom
/bin/bash: line 1: conda: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 55.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.1 MB/s eta 0:00:00
  Created wheel for jieba_fast: filename=jieba_fast-0.53-cp312-cp312-linux_x86_64.whl size=7659510 sha256=8652829f07e499697c3d5e02dc48f5c9ab671b5125226eef71c6a51a352d031c
  Stored in directory: /root/.cache/pip/wheels/65/67/37/8968a5b150cd26683fd229f2987694f1ff76c035f9ddc84bfe
Successfully built jieba_fa

In [2]:
import torch
import subprocess

# Kiểm tra CUDA
print("=== Kiểm tra CUDA ===")
print("CUDA có sẵn để training không?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dòng GPU đang sử dụng:", torch.cuda.get_device_name(0))

# Kiểm tra Conda
print("\n=== Kiểm tra Conda ===")
try:
    conda_version = subprocess.check_output(["conda", "--version"]).decode('utf-8').strip()
    print("Thông tin Conda:", conda_version)
except FileNotFoundError:
    print("Conda KHÔNG được cài đặt (Điều này là bình thường trên Kaggle, vì hệ thống dùng pip làm trình quản lý gói mặc định).")

=== Kiểm tra CUDA ===
CUDA có sẵn để training không?: True
Dòng GPU đang sử dụng: Tesla T4

=== Kiểm tra Conda ===
Conda KHÔNG được cài đặt (Điều này là bình thường trên Kaggle, vì hệ thống dùng pip làm trình quản lý gói mặc định).


In [3]:
!apt-get update && apt-get install -y p7zip-full
!cd data && 7z x GeoGLUE_clean.7z -y
!python model/dataset.py --dataset GeoGLUE_clean

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main 

In [4]:
import re

file_path = 'model/geobloom_v19.py'
with open(file_path, 'r') as f:
    content = f.read()

# Tìm và thay thế tất cả các biến num_workers thành 0
new_content = re.sub(r'num_workers\s*=\s*\d+', 'num_workers=0', content)

with open(file_path, 'w') as f:
    f.write(new_content)
print("Đã ép num_workers về 0 để tiết kiệm RAM.")

Đã ép num_workers về 0 để tiết kiệm RAM.


In [5]:
!grep "num_workers" model/geobloom_v19.py

            train_dataloader.append(DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, collate_fn=train_dataset.collate_fn))


In [6]:
!python model/geobloom_v19.py --dataset GeoGLUE_clean --epochs 5

[1/2] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output isin_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=isin_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -lineinfo -std=c++17 -c /kaggle/working/GeoBloom/cuda/isin_cuda.cu -o isin_cuda.cuda.o 
[2/2] c++ isin_cuda.cuda.o -shared -L/usr/local/lib/python3.12/dist-packages/torch/lib -lc10 -lc10_cuda -ltorch_cpu -ltorch_cuda -ltorch -ltorch_python -L/usr/local/cuda/lib64 -lcudart -o isin_cuda.so
/kaggle/working/GeoBloom/model/geobloom_v19.py:467: FutureWarning: `to

In [7]:
!nnue/v19/nnue GeoGLUE_clean test 8 800-800-800-800

Beam width: 800 800 800 800 Loading dataset from path: data_bin/GeoGLUE_clean/
Allocating 277.549 MB for 142105 bloom filters on the tree...
Allocating 94.8846 MB for 777295 leaf nodes at depth 0...
Infering test candidates...
Searching 10000th query...
Total search time of all threads: 106.409s, Query Per Second: 114.248
=============== Intermediate Recall Scores ==============
0.988320	0.962738	0.934852	0.896603	
====================== Evaluation =======================
Recall@20 	 Recall@10  	 NDCG@5  	 NDCG@1
0.787941	0.732582	0.550498	0.421239
Predictions saved to data_bin/GeoGLUE_clean/test_nodes.bin
